In [0]:
dbutils.widgets.removeAll() 

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_dev")
dbutils.widgets.text("source", "silver")
dbutils.widgets.text("sink", "golden")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
schema_source = dbutils.widgets.get("source")
schema_sink = dbutils.widgets.get("sink")

## 1. Lectura de tabla fuente Silver

In [0]:
df_silver = spark.table(f"{catalogo}.{schema_source}.ventas_productos_categorias")
display(df_silver)

## 2. Transformación: agregación diaria por tienda

In [0]:
df_ventas_diarias = (
    df_silver
    .groupBy(
        F.col("TIENDA"),
        F.col("FECHA")
    )
    .agg(
        F.countDistinct("ID_PEDIDO").alias("NUM_PEDIDOS"),
        F.sum("CANTIDAD").alias("TOTAL_UNIDADES"),
        F.round(
            F.sum(F.col("PRECIO") * F.col("CANTIDAD")), 2
        ).alias("INGRESO_BRUTO"),
        F.round(
            F.sum(
                F.col("PRECIO") * F.col("CANTIDAD") * (1 - F.col("DESCUENTO") / 100)
            ), 2
        ).alias("INGRESO_NETO"),
        F.round(
            F.avg(
                F.when(F.col("DESCUENTO") > 0, F.col("DESCUENTO"))
            ), 2
        ).alias("DESCUENTO_MEDIO_PCT"),
        F.count(
            F.when(F.col("DESCUENTO") > 0, 1)
        ).alias("NUM_LINEAS_CON_DESCUENTO"),
        F.count("*").alias("NUM_LINEAS_TOTAL")
    )
    .withColumn(
        "PCT_LINEAS_CON_DESCUENTO",
        F.round(
            F.col("NUM_LINEAS_CON_DESCUENTO") / F.col("NUM_LINEAS_TOTAL") * 100, 2
        )
    )
    .withColumn(
        "FECHA_ACTUALIZACION",
        F.current_timestamp()
    )
)

display(df_ventas_diarias)

## 3. Cast final y escritura en Gold

In [0]:
target_table = f"{catalogo}.{schema_sink}.ventas_diarias_tienda"

(
    df_ventas_diarias
    .select(
        F.col("TIENDA").cast("string").alias("TIENDA"),
        F.col("FECHA").cast("date").alias("FECHA"),
        F.col("NUM_PEDIDOS").cast("int").alias("NUM_PEDIDOS"),
        F.col("TOTAL_UNIDADES").cast("int").alias("TOTAL_UNIDADES"),
        F.col("INGRESO_BRUTO").cast("double").alias("INGRESO_BRUTO"),
        F.col("INGRESO_NETO").cast("double").alias("INGRESO_NETO"),
        F.col("DESCUENTO_MEDIO_PCT").cast("double").alias("DESCUENTO_MEDIO_PCT"),
        F.col("NUM_LINEAS_CON_DESCUENTO").cast("int").alias("NUM_LINEAS_CON_DESCUENTO"),
        F.col("NUM_LINEAS_TOTAL").cast("int").alias("NUM_LINEAS_TOTAL"),
        F.col("PCT_LINEAS_CON_DESCUENTO").cast("double").alias("PCT_LINEAS_CON_DESCUENTO"),
        F.col("FECHA_ACTUALIZACION").cast("timestamp").alias("FECHA_ACTUALIZACION")
    )
    .coalesce(4)
    .write
    .mode("overwrite")
    .insertInto(target_table)
)

## 4. Validación

SELECT count(*) FROM catalog_dev.golden.ventas_diarias_tienda;